# RoadTwin — Final DemoReproduces the benchmark end to end and prints the numbers that go on theslide. Run it once before the demo so the outputs are saved in the notebook.**Prerequisites:** SUMO installed with `SUMO_HOME` set, and either networkaccess or a cached OSM extract in `assets/benchmark/`.See `01_pipeline_walkthrough.ipynb` for the explanation of each step.

In [ ]:
import sys, json, subprocessfrom pathlib import PathROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))import config as Cprint("RoadTwin benchmark")print("=" * 60)print(f"location   {C.BENCHMARK['name']}")print(f"coords     {C.BENCHMARK['lat']}, {C.BENCHMARK['lon']}")print(f"AOI        {C.BENCHMARK['aoi_radius_m']} m")print(f"demand     period={C.SIM['period']}  ({C.SIM['begin']}-{C.SIM['end']}s)")print(f"seeds      {C.SIM['seeds']}")

## 1. Environment checkNever demo from an unverified environment.

In [ ]:
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "verify_environment.py")],                   capture_output=True, text=True)print(r.stdout[-2500:])

In [ ]:
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "selftest.py")],                   capture_output=True, text=True)print(r.stdout[-700:])assert r.returncode == 0, "self-tests failing -- fix before demoing"

## 2. Full pipeline```location -> Overpass -> netconvert -> plain XML -> netconvert         -> network.net.xml + road_network.xodr -> randomTrips         -> SUMO baseline -> SUMO closure -> metrics -> ZIP```

In [ ]:
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "run_benchmark.py")],                   capture_output=True, text=True)print(r.stdout[-6000:])if r.returncode != 0:    print("\n--- stderr ---\n", r.stderr[-3000:])

## 3. The resultThese numbers go on the slide. They come from parsing SUMO output — nothinghere is hardcoded.

In [ ]:
proj = C.PROJECTS_DIR / "benchmark"metrics = json.loads((proj / "metrics.json").read_text())b, cl = metrics["baseline"], metrics["closure"]cmp = metrics["comparison"]print(f"{'':<24}{'BASELINE':>12}{'CLOSURE':>12}{'DELTA':>12}")print("-" * 60)for r in cmp["rows"]:    d = "-" if r["delta_pct"] is None else f"{r['delta_pct']:+.1f}%"    print(f"{r['metric']:<24}{r['baseline']:>12}{r['scenario']:>12}{d:>12}")print("-" * 60)print(f"mean of {b['n_seeds']} seeds   "      f"(travel time sd: baseline {b.get('avg_travel_time_s_sd')}s, "      f"closure {cl.get('avg_travel_time_s_sd')}s)")print()print("significant:", cmp["significant"])print(cmp["verdict"])

In [ ]:
# Do not demo a non-significant result.if not cmp["significant"]:    print("!! The closure did not move the metrics beyond seed noise.")    print("   Lower SIM['period'] in config.py and re-run before the demo.")else:    print("Result is larger than seed-to-seed variation. Safe to present.")

## 4. What was closedThe reported closure length is the **actual** edge length. SUMO closes a lanefor a whole edge — there is no partial-length closure — so we report what wasreally modelled rather than a round number.

In [ ]:
print(json.dumps(json.loads((proj / "scenario.json").read_text()), indent=2))

## 5. Artifacts and provenanceEvery artifact records what produced it, from what, with which tool version andwith input/output hashes. This is what makes "auditable" a fact rather than aslogan.

In [ ]:
for f in sorted(proj.rglob("*")):    if f.is_file() and "seed_" not in str(f):        print(f"  {f.relative_to(proj)!s:<44}{f.stat().st_size/1024:8.1f} KB")

In [ ]:
prov = json.loads((proj / "provenance.json").read_text())for e in prov:    ins = ", ".join(Path(i["path"]).name for i in e.get("inputs", [])) or "-"    outs = ", ".join(Path(o["path"]).name for o in e.get("outputs", [])) or "-"    print(f"{e['source']:<12} {e['tool_version'][:38]:<40}")    print(f"   in : {ins}")    print(f"   out: {outs}")

## 6. The OpenDRIVEWritten by `netconvert --opendrive-output`, and re-imported through netconvertto prove it is valid — step 7 of the run above prints PASS.**Stated limitation:** the export carries geometry, lanes and junctions. Signaldata is not represented in the `.xodr`; it lives in `sumo/plain.tll.xml` and in`roadtwin.json`. The information is not lost, it is not in this file.

In [ ]:
xodr = proj / "road_network.xodr"print(f"{xodr.name}  {xodr.stat().st_size/1024:.1f} KB\n")print(xodr.read_text()[:1400])

## 7. Deliverable`RoadTwin_Project.zip` — the model, the OpenDRIVE, the observations, thevalidation report, the simulation inputs and outputs, the provenance chain, anda README generated from this run.

In [ ]:
import zipfilezp = C.PROJECTS_DIR / "RoadTwin_Project_benchmark.zip"print(f"{zp.name}  {zp.stat().st_size/1e6:.2f} MB\n")for n in sorted(zipfile.ZipFile(zp).namelist()):    print("  ", n)

---## Demo checklistBefore you present, confirm every line:```[ ] verify_environment.py    0 FAIL[ ] selftest.py              0 failed[ ] run_benchmark.py         completed, significant == true[ ] OpenDRIVE round-trip     PASS[ ] cached OSM committed[ ] cached mosaic + vision outputs present[ ] backup demo video recorded, stored locally[ ] pre-built ZIP on disk as a fallback[ ] app installed and whitelisted on the demo machine[ ] SUMO_HOME set on the demo machine[ ] rehearsed three times, timed[ ] laptop plugged in, GPU on the performance profile```The moment that matters most: **accept a piece of AI evidence and watch thetraffic numbers change.** Rehearse that transition specifically — it is whatmakes the twin feel real rather than decorative.